In [91]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import json
import sklearn
import numpy as np

# Reading the graph

In [20]:
df = pd.read_csv("lasftm_asia/lastfm_asia_edges.csv", dtype={"node_1": str, "node_2": str})

G = nx.from_pandas_edgelist(df, source="node_1", target="node_2")

with open("lasftm_asia/lastfm_asia_features.json", "r") as f:
    features = json.load(f)

for node, feat in features.items():
    if node in G:
        G.nodes[node]["feat"] = feat

df = pd.read_csv("lasftm_asia/lastfm_asia_target.csv", dtype={"id": str})

target_map = dict(zip(df["id"], df["target"]))

nx.set_node_attributes(G, target_map, "target")


In [110]:
import random
def split_graph(G, ratio=0.8):
    n = list(G.nodes)
    ln = len(n)
    train_nodes = random.sample(n, int(ln*ratio))
    test_nodes = set(n).difference(train_nodes)
    return list(train_nodes), list(test_nodes)

In [113]:
from collections import Counter



def classify_node(G, node_id, train, global_majority):

    
    
    neighbor_labels = [
        G.nodes[n]["target"]
        for n in G.neighbors(node_id)
        if "target" in G.nodes[n]
        and n in train
    ]

    if len(neighbor_labels) == 0:
        return global_majority

    return Counter(neighbor_labels).most_common(1)[0][0]

In [114]:
from sklearn.metrics import accuracy_score, f1_score

accuracies = []
f1s = []

for _ in range(100):
    train, test = split_graph(G)

    global_majority = Counter(
    G.nodes[n]["target"] for n in train if "target" in G.nodes[n]
    ).most_common(1)[0][0]

    ground_truth = [G.nodes[node]["target"] for node in test]
    preds = [classify_node(G, node, train, global_majority) for node in test]
    acc = accuracy_score(ground_truth, preds)
    accuracies.append(acc)
    f1 = f1_score(ground_truth, preds, average="macro")
    f1s.append(f1)

print(f"Accuracy: {np.mean(accuracies)} +/- {np.std(accuracies)}")
print(f"F1: {np.mean(f1s)} +/- {np.std(f1s)}")

Accuracy: 0.8445901639344261 +/- 0.008496296045505843
F1: 0.781013455151709 +/- 0.013365151468427188


In [119]:
from collections import Counter

def label_propagation(G, train, max_iter=10):
    # initialize
    labels = {
        n: G.nodes[n].get("target", None)
        for n in G.nodes
    }

    for _ in range(max_iter):
        new_labels = labels.copy()

        for n in G.nodes:
            if n in train:
                continue

            neighbor_labels = [
                labels[nbr]
                for nbr in G.neighbors(n)
                if nbr in train and labels[nbr] is not None
            ]

            if len(neighbor_labels) > 0:
                new_labels[n] = Counter(neighbor_labels).most_common(1)[0][0]

        labels = new_labels

    return labels

In [120]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter

accuracies = []
f1s = []

for _ in range(100):
    train, test = split_graph(G)

    pred = label_propagation(G, train)

    ground_truth = [G.nodes[n]["target"] for n in test]
    preds = [pred[n] for n in test]

    acc = accuracy_score(ground_truth, preds)
    f1 = f1_score(ground_truth, preds, average="macro")

    accuracies.append(acc)
    f1s.append(f1)

print(f"Accuracy: {np.mean(accuracies)} +/- {np.std(accuracies)}")
print(f"F1: {np.mean(f1s)} +/- {np.std(f1s)}")

Accuracy: 0.8848393442622953 +/- 0.007472406644372021
F1: 0.822832028337631 +/- 0.019324879737543498
